# BENV0169 Assessment 1 — Task 1

## Data quality and energy forecasting for building operation

You are advising an education building that plans to use flexible electrical loads, rooftop PV and battery storage. You will check the input data, reconstruct missing meter readings, compare day-ahead load forecasts, assess hourly PV forecasts and evaluate PV forecast uncertainty.

Complete the marked code and written-answer cells. Keep the supplied names, dates, features, model settings and random seed. Do not hard-code gap locations or expected scores. Calculate with unrounded values. Display load results to two decimal places, PV results to four and coverage to one.

The written answers have a combined limit of **750 words**: Q1 80, Q2 130, Q3 90, Q4 150, Q5 140 and Q6 160 words. The questions are worth 10, 15, 15, 20, 20 and 20 marks, respectively.

Assume that a measurement is available at the timestamp shown in its index. Keep the supplied building timestamps unchanged. PV timestamps are in UTC. Do not apply your own daylight-saving conversion.

## Setup

Run this notebook from the extracted assessment folder. The file paths are already supplied. Public checks use small artificial examples. They provide feedback but carry no marks and do not prove that the whole solution is correct.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from public_checks import (
    check_imputation_function,
    check_interval_metrics,
    check_metric_functions,
    check_unit_interval_projection,
)
from task1_config import *

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_rows", 100)

## Q1 — Audit and prepare the building data (10 marks; 80 words)

Use `data/building_load.csv`. Keep the original data unchanged and make a working copy.

1. Count duplicate timestamp rows. Remove the extra rows and keep the first occurrence.
2. After removing duplicates, count the stored missing values in each measurement column.
3. Sort the timestamps and create a row for every hour from the first timestamp to the last. Count the timestamps that were absent from the original data separately.
4. Calculate the percentage of missing electricity readings using all expected hourly rows as the denominator.
5. Return `quality_summary`, `missing_blocks` and `building_clean`. For each missing block, report its inclusive `start`, inclusive `end` and `duration_hours`. Do not fill the Q1 gaps.

In Q3, create lagged variables on this complete hourly grid **before** removing incomplete training rows.

In [ ]:
building_raw = pd.read_csv(DATA_DIR / "building_load.csv")

def find_missing_blocks(missing_mask: pd.Series) -> pd.DataFrame:
    """Return start, end and duration_hours for each consecutive True block."""
    # TODO: detect each run of True values without hard-coding timestamps.
    raise NotImplementedError("Complete find_missing_blocks for Q1")

def audit_building_data(raw: pd.DataFrame):
    """Return quality_summary, missing_blocks and hourly building_clean."""
    # TODO: parse timestamps; count/remove duplicate timestamps; count stored
    # NaNs; sort; create the expected hourly index; count absent timestamps;
    # reindex; calculate the missing percentage; and call find_missing_blocks.
    raise NotImplementedError("Complete audit_building_data for Q1")

quality_summary, missing_blocks, building_clean = audit_building_data(building_raw)
display(quality_summary)
display(missing_blocks)

### Q1 written answer (maximum 80 words)

Report the missing-electricity percentage and the longest gap. Explain how duplicate timestamps and absent hours affect lagged predictors. Why can removing incomplete rows before creating lags produce incorrect inputs?

**Answer:**  
_Replace this text._

## Q2 — Evaluate reconstruction of missing readings (15 marks; 130 words)

Use the period from 1 April 2016 00:00 to 5 May 2016 23:00. `imputation_reference.csv` contains the complete observed values. Use it only for scoring and plotting.

Reindex the gapped series onto the complete hourly time grid before detecting and reconstructing the gaps. Derive `imputation_mask` and `imputation_blocks` from this gapped series before filling it. Do not use the reference file to locate the gaps.

Implement four methods: forward fill, the mean of the observed values in the gapped series, time-based linear interpolation, and the value exactly 168 hours earlier. Each function must receive only the gapped series, preserve its observed values and return the complete hourly index.

Score only the detected missing timestamps. Group results by gap duration and give every missing observation equal weight. Produce Table 1 (12 rows) and Figure 1 (three panels showing the earliest 1-, 6- and 24-hour gaps).

In [ ]:
q2_reference = pd.read_csv(DATA_DIR / "imputation_reference.csv", parse_dates=["timestamp"])
q2_gapped_raw = pd.read_csv(DATA_DIR / "imputation_gapped.csv", parse_dates=["timestamp"])

# TODO: put the reference and gapped electricity series on the prescribed hourly
# index. Derive imputation_mask and imputation_blocks from q2_gapped only.
q2_grid = pd.date_range(Q2_START, Q2_END, freq="h")
q2_reference = ...
q2_gapped = ...
imputation_mask = ...
imputation_blocks = ...

# TODO: assert index alignment and equality of all retained observations.

def _aligned_arrays(actual, forecast):
    """Validate length, pandas alignment, dimensionality and finite values."""
    raise NotImplementedError("Complete strict metric input validation")

def mae(actual, forecast) -> float:
    """Return mean absolute error after strict input validation."""
    raise NotImplementedError("Complete MAE; reuse it in Q4")

def rmse(actual, forecast) -> float:
    """Return root mean squared error after strict input validation."""
    raise NotImplementedError("Complete RMSE; reuse it in Q4")

def impute_forward_fill(series: pd.Series) -> pd.Series:
    """Fill gaps using the most recent observed value; preserve observations."""
    raise NotImplementedError("Complete forward filling for Q2")

def impute_observed_mean(series: pd.Series) -> pd.Series:
    """Fill gaps using the mean of observations present in the input."""
    raise NotImplementedError("Complete mean reconstruction for Q2")

def impute_linear(series: pd.Series) -> pd.Series:
    """Fill internal gaps using time-based linear interpolation."""
    raise NotImplementedError("Complete time interpolation for Q2")

def impute_previous_week(series: pd.Series) -> pd.Series:
    """Fill gaps from the original value exactly 168 hours earlier."""
    raise NotImplementedError("Complete previous-week reconstruction for Q2")

# Optional feedback on hand-checkable examples (uncomment after implementing).
# check_imputation_function(impute_forward_fill, [1, 1, 1, 4, 5])
# check_imputation_function(impute_observed_mean, [1, 10/3, 10/3, 4, 5])
# check_imputation_function(impute_linear, [1, 2, 3, 4, 5])

imputations = {
    "Forward fill": impute_forward_fill(q2_gapped),
    "Observed mean": impute_observed_mean(q2_gapped),
    "Linear interpolation": impute_linear(q2_gapped),
    "Previous week": impute_previous_week(q2_gapped),
}

# TODO: verify completeness and preservation of observations. Associate every
# missing timestamp with its detected gap duration, then create the 12-row
# table_1 using mae() and rmse() after you implement them in Q4.
table_1 = ...
display(table_1)

# TODO: create Figure 1. For each duration, select the earliest detected block,
# extend 24 h before its start and 24 h after its end, plot truth/gapped/four
# reconstructions, and shade the missing interval.

### Q2 written answer (maximum 130 words)

Compare the errors for different gap lengths. Does the method with the lowest error also preserve the daily shape and peaks? Recommend a method for short gaps and one for long gaps. Which methods need observations from after the gap? Can results from retrospective cleaning justify using those methods during a live outage?

**Answer:**  
_Replace this text. Insert Table 1 and Figure 1 in the answer form._

## Q3 — Formulate the day-ahead load forecast (15 marks; 90 words)

At 23:00 each day, forecast every hour of the next day, from 00:00 to 23:00. These are leads 1–24. Move the forecast origin forward by 24 hours for the next day. Keep the fitted model parameters fixed, but use new observations when they become available for later forecast origins.

Use 8 January–30 September 2016 for training and 1–28 October 2016 for testing. On the complete hourly grid, create `load_lag_24`, `load_lag_168` and `temperature_persistence_c`. Then add the specified integer and cyclical calendar variables.

Remove incomplete **training** rows only after creating the features. Keep all 672 test hours. Create the origin, lead and source-timestamp records, and produce Figure 2a.

In [ ]:
def build_load_features(building_clean: pd.DataFrame) -> pd.DataFrame:
    """Construct the prescribed lag and calendar features on the hourly grid."""
    # TODO: add load and observed-temperature lags; integer hour/day/weekend;
    # and sine/cosine terms with periods 24 and 7.
    raise NotImplementedError("Complete load feature construction for Q3")

def build_load_origin_records(test_index: pd.DatetimeIndex) -> pd.DataFrame:
    """Return target, origin, lead and source timestamps; assert availability."""
    # TODO: origin is 23:00 before each target day; leads are 1...24.
    raise NotImplementedError("Complete load origin/lead records for Q3")

load_features = build_load_features(building_clean)

# TODO: select candidate training/test rows; count missing values per required
# field; remove incomplete training rows; assert 672 complete test rows; create
# load_split_summary and load_origin_lead_records.
train_candidate = ...
train_load = ...
test_load = ...
load_split_summary = ...
load_origin_lead_records = ...
display(load_split_summary)

# TODO: Figure 2a — mean observed training electricity by hour, separately for
# weekdays and weekends. Exclude only missing target values from this profile.

### Q3 written answer (maximum 90 words)

Explain how a day-ahead forecast supports scheduling. For a target at 15:00 tomorrow, give the timestamps used for the two load lags and the temperature proxy. Why would a one-hour load lag and tomorrow's observed weather be unavailable at today's 23:00 forecast origin?

**Answer:**  
_Replace this text. Insert the combined Figure 2 after completing Q5._

## Q4 — Compare load forecasting models (20 marks; 150 words)

Compare four models: daily seasonal naïve (`load_lag_24`), weekly seasonal naïve (`load_lag_168`), Ridge regression and random forest. Ridge uses both load lags, temperature persistence, the hour/day sine and cosine variables, and the weekend indicator. The random forest uses both load lags, temperature persistence, integer hour, integer day of week and the weekend indicator.

Fit each model once using training data only. Set negative load forecasts to zero before scoring and plotting.

Implement MAE and RMSE. The functions must reject unequal lengths, mismatched pandas indices and non-finite values instead of silently removing them. Produce Table 2a, `load_scores_by_lead` (96 rows) and Figure 3a for 10–16 October. With a fixed 23:00 origin, each lead is also tied to a particular clock hour.

In [ ]:
def clip_nonnegative(values: pd.Series) -> pd.Series:
    """Project load forecasts onto the non-negative range."""
    raise NotImplementedError("Complete non-negative load projection for Q4")

# Optional feedback (uncomment after implementing metrics).
# check_metric_functions(mae, rmse)

ridge_columns = [
    "load_lag_24", "load_lag_168", "temperature_persistence_c",
    "hour_sin", "hour_cos", "day_sin", "day_cos", "is_weekend",
]
forest_columns = [
    "load_lag_24", "load_lag_168", "temperature_persistence_c",
    "hour", "day_of_week", "is_weekend",
]

# TODO: instantiate the prescribed models, fit on train_load only, generate the
# four indexed test forecasts, and apply clip_nonnegative before scoring.
ridge = make_pipeline(StandardScaler(), Ridge(alpha=10.0))
forest = RandomForestRegressor(**RF_KWARGS)
load_predictions = ...

# TODO: create table_2a with full/weekday/weekend scores and create the 96-row
# load_scores_by_lead table using the origin records. Plot Figure 3a.
table_2a = ...
load_scores_by_lead = ...
display(table_2a)

### Q4 written answer (maximum 150 words)

Compare each model with the weekly benchmark using numbers from your results. Do MAE and RMSE give the same ranking? Explain why the rankings may differ, or what their agreement shows. Recommend one model, considering both typical errors and large misses. Describe one weekday/weekend difference and one lead-specific pattern. Remember that lead and clock hour coincide in this experiment.

**Answer:**  
_Replace this text. Insert the combined Table 2 and Figure 3 after completing Q5._

## Q5 — Implement one-step-ahead PV forecasts (20 marks; 140 words)

For each valid time $t$, forecast PV from the origin $t-1$ hour. Move the origin forward by one hour each time and update the recent PV inputs. Do not refit the models during calibration or testing. Use the supplied UTC training, calibration and test periods.

Compare persistence, one linear regression and one random forest. The two fitted models use the same inputs: `pv_lag_1`, `pv_lag_24`, forecast irradiance, forecast temperature, forecast cloud fraction, and sine/cosine variables for UTC hour and day of year.

Check that each weather forecast was issued by the PV forecast origin and that its recorded lead agrees with its issue and valid times. Limit every PV point forecast to $[0,1]$ p.u. Do not force forecasts to zero during fixed night-time hours. Produce Table 2b, `pv_scores_by_hour`, Figure 2b and Figure 3b.

In [ ]:
pv = pd.read_csv(DATA_DIR / "pv_forecasts.csv")
pv["timestamp_utc"] = pd.to_datetime(pv["timestamp_utc"], utc=True)
pv["forecast_issue_time_utc"] = pd.to_datetime(pv["forecast_issue_time_utc"], utc=True)
pv = pv.sort_values("timestamp_utc").set_index("timestamp_utc")

def build_pv_features(pv: pd.DataFrame) -> pd.DataFrame:
    """Construct prescribed PV lags and UTC calendar features on the full grid."""
    raise NotImplementedError("Complete PV feature construction for Q5")

def pv_origin_records(features: pd.DataFrame) -> pd.DataFrame:
    """Return PV origin/source records and assert lag/weather availability."""
    raise NotImplementedError("Complete PV availability checks for Q5")

def clip_unit_interval(values: pd.Series) -> pd.Series:
    """Project PV point forecasts or interval endpoints onto [0, 1]."""
    raise NotImplementedError("Complete the physical PV bound for Q5")

# Optional feedback (uncomment after implementing the projection).
# check_unit_interval_projection(clip_unit_interval)

pv_features = build_pv_features(pv)
pv_records = pv_origin_records(pv_features)
pv_model_columns = [
    "pv_lag_1", "pv_lag_24",
    "forecast_surface_solar_radiation_w_m2",
    "forecast_air_temperature_c",
    "forecast_total_cloud_cover_fraction",
    "hour_sin", "hour_cos", "doy_sin", "doy_cos",
]

# TODO: create train/calibration/test frames; remove incomplete training rows
# only; fit one LinearRegression and one forest on training data; forecast all
# test hours along with persistence; apply [0,1] projection.
pv_train = ...
pv_cal = ...
pv_test = ...
linear_pv = LinearRegression(fit_intercept=True)
forest_pv = RandomForestRegressor(**RF_KWARGS)
pv_predictions = ...

# TODO: create table_2b and the 72-row pv_scores_by_hour. Add Figure 2b (training
# PV versus forecast irradiance) and Figure 3b (12–18 August 2013 UTC).
table_2b = ...
pv_scores_by_hour = ...
display(table_2b)

### Q5 written answer (maximum 140 words)

Compare the fitted models with persistence using the full-test and hourly results. Why is the most recent PV reading available for each one-hour forecast but not for every hour of a fixed day-ahead forecast? Explain the physical limits on PV forecasts. Why can full-day scores and MAPE give a misleading view of PV performance?

**Answer:**  
_Replace this text. Complete the combined Figures 2–3 and Table 2 in the answer form._

## Q6 — Evaluate PV forecast uncertainty (20 marks; 160 words)

Use the fixed linear model from Q5. Estimate residual quantiles from the calibration period only. Compare two methods for a nominal 90% interval:

- one pair of 5th and 95th residual percentiles for all hours;
- a separate percentile pair for each UTC hour.

Use `numpy.quantile(..., method="linear")`. Add the relevant residual quantiles to the same bounded test point forecast, then limit both interval endpoints to $[0,1]$.

Implement inclusive coverage, mean interval width and the 90% Winkler score. Produce Table 3 (two rows), `pv_interval_scores_by_hour` (48 rows), the residual-quantile table with sample counts, and Figure 4 for 15–17 August 2013 UTC. Use the same axes in both panels.

In [ ]:
def interval_coverage(actual, lower, upper) -> float:
    """Return inclusive empirical interval coverage as a percentage."""
    raise NotImplementedError("Complete inclusive coverage for Q6")

def winkler_score(actual, lower, upper, alpha: float = 0.10) -> float:
    """Return mean Winkler score for central (1-alpha) intervals."""
    raise NotImplementedError("Complete the Winkler score for Q6")

# Optional feedback (uncomment after implementing interval metrics).
# check_interval_metrics(interval_coverage, winkler_score)

# TODO: forecast the calibration set with the already-fitted linear model;
# calculate actual-minus-forecast residuals; estimate pooled and hourly q05/q95
# using calibration only; retain counts in residual_quantiles.
pv_cal_prediction = ...
calibration_residuals = ...
residual_quantiles = ...

# TODO: construct pooled/hourly test bounds around the same bounded linear point
# forecast, project endpoints to [0,1], and assert lower <= upper.
intervals = ...

# TODO: create two-row table_3, 48-row pv_interval_scores_by_hour and Figure 4.
table_3 = ...
pv_interval_scores_by_hour = ...
display(table_3)

### Q6 written answer (maximum 160 words across parts a–b)

**Part a.** Use Table 3, Figure 4 and one hourly result to compare the pooled and hourly intervals. Discuss coverage against the 90% target, mean width and Winkler score. Recommend one method. Explain why narrow intervals or high overall coverage alone are not enough.

**Part b.** Explain the difference between a point forecast, a conditional quantile, a marginal interval and a trajectory scenario. Why is a sequence of separately updated one-hour intervals not a coherent multi-hour trajectory for operational optimisation?

**Answer:**  
_Replace this text. Insert Table 3 and Figure 4 in the answer form._

## Submission checklist

- Restart the kernel and run every completed cell in order. Keep the outputs.
- Check that all required tables, figures and diagnostic data frames are present.
- Use unrounded values in calculations and round only for display.
- Use relative paths only. The notebook must not depend on private files.
- Complete the answer form without exceeding the six word limits.
- Submit the executed notebook and combined coursework PDF as stated on Moodle.